# Automatic Chord Recognition

End-to-end pipeline on the tinyAAM dataset (20 tracks).  
Features: log-frequency chromagrams → context windows → MLP classifier → 25 chord classes.

## Section 1 — Setup & Exploration

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.data.load_data import AAMDataset, load_training_example
from src.features.vocab import ChordVocab, CHORD_VOCAB, build_vocab_from_dataset
from src.features.chroma import (
    prepare_track_features, normalize_chroma, median_filter_chroma, build_context_windows
)
from src.data.dataset import ChordDataset, create_dataloaders
from src.models.mlp import ChordMLP, ChordContextMLP
from src.training.trainer import train, evaluate, save_checkpoint, load_checkpoint
from src.training.metrics import frame_accuracy, per_class_accuracy, confusion_matrix_df
from src.utils.visualization import (
    plot_chroma, plot_chord_prediction, plot_training_curves, plot_confusion_matrix
)

vocab = ChordVocab.from_default()
print(f'Vocabulary ({vocab.num_classes} classes):', CHORD_VOCAB)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Verify dataset vocabulary matches hardcoded constant
dataset_vocab = build_vocab_from_dataset(root='../data/raw/tinyAAM')
print('Dataset vocab:', dataset_vocab)
assert set(dataset_vocab) == set(CHORD_VOCAB), 'Vocab mismatch!'
print('Vocab verified!')


In [ ]:
# List all tracks
ds = AAMDataset(root='../data/raw/tinyAAM')
all_track_ids = ds.track_ids()
print(f'Total tracks: {len(all_track_ids)}')
print('Track IDs:', all_track_ids)


In [ ]:
# Chord distribution across all tracks
label_counts = {c: 0 for c in CHORD_VOCAB}
for tid in all_track_ids:
    ex = load_training_example(tid, root='../data/raw/tinyAAM')
    for lbl in ex['frame_labels']:
        if lbl in label_counts:
            label_counts[lbl] += 1

count_df = pd.Series(label_counts).sort_values(ascending=False)
count_df.plot(kind='bar', figsize=(14, 4), title='Chord Frame Counts (all tracks)')
plt.tight_layout()
plt.show()
print(count_df)


In [ ]:
# Chroma + chord overlay for track 0001
ex0 = load_training_example('0001', root='../data/raw/tinyAAM')
chroma0 = ex0['chroma']
times0  = ex0['frame_times']
labels0 = ex0['frame_labels']

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
plot_chroma(chroma0, sr=22050, hop_length=512, title='Track 0001 — Chromagram', ax=axes[0])
prev = None
for i, lbl in enumerate(labels0):
    if lbl != prev:
        axes[0].axvline(times0[i], color='red', alpha=0.4, linewidth=0.6)
        prev = lbl
unique_chords, idx = np.unique(labels0, return_inverse=True)
axes[1].plot(times0, idx, drawstyle='steps-post')
axes[1].set_yticks(range(len(unique_chords)))
axes[1].set_yticklabels(unique_chords, fontsize=7)
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Chord labels over time')
plt.tight_layout()
plt.show()


## Section 2 — Feature Preparation

In [ ]:
# Normalization comparison
chroma_raw = ex0['chroma']
chroma_l2  = normalize_chroma(chroma_raw, method='l2')
chroma_max = normalize_chroma(chroma_raw, method='max')
chroma_med = median_filter_chroma(chroma_l2, kernel_size=5)

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
plot_chroma(chroma_raw, title='Raw chroma', ax=axes[0])
plot_chroma(chroma_l2,  title='L2-normalized', ax=axes[1])
plot_chroma(chroma_med, title='L2 + Median filter (k=5)', ax=axes[2])
plt.tight_layout()
plt.show()


In [ ]:
# Context window shape demo
CONTEXT = 7
windows = build_context_windows(chroma_l2, context=CONTEXT)
print(f'Context window shape: {windows.shape}  (T, 12*(2*{CONTEXT}+1)={12*(2*CONTEXT+1)})')

plt.figure(figsize=(12, 3))
plt.imshow(windows[:200].T, aspect='auto', origin='lower', cmap='RdYlBu_r')
plt.colorbar()
plt.title('Context windows — first 200 frames, 180 features')
plt.xlabel('Frame'); plt.ylabel('Feature dim')
plt.tight_layout()
plt.show()


## Section 3 — Build Datasets

In [ ]:
# Fixed splits
TEST_IDS  = ['2990', '3000']
VAL_IDS   = ['2828', '2841']
TRAIN_IDS = [tid for tid in all_track_ids if tid not in TEST_IDS + VAL_IDS]

print(f'Train: {len(TRAIN_IDS)} tracks -> {TRAIN_IDS}')
print(f'Val:   {len(VAL_IDS)} tracks  -> {VAL_IDS}')
print(f'Test:  {len(TEST_IDS)} tracks  -> {TEST_IDS}')


In [ ]:
ROOT = '../data/raw/tinyAAM'
BATCH_SIZE = 256

print('Loading context datasets (may take a minute)...')
train_loader, val_loader, test_loader = create_dataloaders(
    TRAIN_IDS, VAL_IDS, TEST_IDS,
    root=ROOT, vocab=vocab, context=CONTEXT, batch_size=BATCH_SIZE,
)
print(f'Train frames: {len(train_loader.dataset):,}')
print(f'Val frames:   {len(val_loader.dataset):,}')
print(f'Test frames:  {len(test_loader.dataset):,}')
print(f'Total:        {len(train_loader.dataset)+len(val_loader.dataset)+len(test_loader.dataset):,}')


## Section 4 — MLP Baseline (no context)

In [ ]:
# Build baseline datasets (context=0 → 12-dim input)
print('Loading baseline datasets...')
train_loader_b, val_loader_b, _ = create_dataloaders(
    TRAIN_IDS, VAL_IDS, TEST_IDS,
    root=ROOT, vocab=vocab, context=0, batch_size=BATCH_SIZE,
)
print('Done.')


In [ ]:
baseline_model = ChordMLP(
    num_classes=vocab.num_classes, hidden_dim=256, num_layers=3, dropout=0.3
)
print(baseline_model)
print(f'Parameters: {sum(p.numel() for p in baseline_model.parameters()):,}')


In [ ]:
history_baseline = train(
    baseline_model, train_loader_b, val_loader_b,
    num_epochs=50, lr=1e-3, device=device, patience=10,
)
print(f"Best epoch: {history_baseline['best_epoch']+1}")
print(f"Best val accuracy: {max(history_baseline['val_accs']):.3f}")


In [ ]:
plot_training_curves(history_baseline)
plt.suptitle('Baseline MLP — Training Curves', y=1.02)
plt.show()


## Section 5 — Context MLP

In [ ]:
context_model = ChordContextMLP(
    context=CONTEXT, num_classes=vocab.num_classes, hidden_dim=256, num_layers=3, dropout=0.3
)
print(context_model)
print(f'Parameters: {sum(p.numel() for p in context_model.parameters()):,}')


In [ ]:
history_context = train(
    context_model, train_loader, val_loader,
    num_epochs=50, lr=1e-3, device=device, patience=10,
)
print(f"Best epoch: {history_context['best_epoch']+1}")
print(f"Best val accuracy: {max(history_context['val_accs']):.3f}")


In [ ]:
plot_training_curves(history_context)
plt.suptitle('Context MLP — Training Curves', y=1.02)
plt.show()

print('--- Model Comparison ---')
print(f"Baseline  best val acc: {max(history_baseline['val_accs']):.3f}")
print(f"Context   best val acc: {max(history_context['val_accs']):.3f}")


## Section 6 — Test Evaluation

In [ ]:
# Load best weights
context_model.load_state_dict(history_context['best_state_dict'])
context_model = context_model.to(device)
context_model.eval()

all_true, all_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = context_model(x).argmax(dim=-1).cpu()
        all_true.append(y.numpy())
        all_pred.append(preds.numpy())

y_true = np.concatenate(all_true)
y_pred = np.concatenate(all_pred)

acc = frame_accuracy(y_true, y_pred)
print(f'Test frame accuracy: {acc:.3f}')


In [ ]:
per_class_df = per_class_accuracy(y_true, y_pred, vocab)
print(per_class_df.sort_values('accuracy', ascending=False).to_string())


In [ ]:
cm_df = confusion_matrix_df(y_true, y_pred, vocab)
try:
    plot_confusion_matrix(cm_df)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install seaborn to display heatmap.')
    print(cm_df)


In [ ]:
# Chord prediction strip on test track 2990
test_ex = load_training_example(TEST_IDS[0], root=ROOT)
chroma_t = test_ex['chroma']
ft       = test_ex['frame_times']
true_lbl = test_ex['frame_labels']

feat_t = prepare_track_features(chroma_t, context=CONTEXT)
x_t    = torch.from_numpy(feat_t).to(device)
with torch.no_grad():
    pred_idx = context_model(x_t).argmax(dim=-1).cpu().numpy()
pred_lbl = vocab.decode(pred_idx)

plot_chord_prediction(ft, true_lbl, pred_lbl, title=f'Track {TEST_IDS[0]} — Chord Predictions')
plt.tight_layout()
plt.show()


In [ ]:
# Save checkpoint with metadata
import os
os.makedirs('../checkpoints', exist_ok=True)
save_checkpoint(
    {
        'model_state': history_context['best_state_dict'],
        'context': CONTEXT,
        'hop_length': 512,
        'sr': 22050,
        'vocab': CHORD_VOCAB,
    },
    '../checkpoints/context_mlp.pt',
)
print('Checkpoint saved to ../checkpoints/context_mlp.pt')


## Section 7 — Error Analysis

### Key Observations

1. **Class imbalance** — Each track contains only a handful of distinct chords, so some classes have very few frames across the whole training set. Weighted cross-entropy partially compensates but rare chords still suffer.

2. **Enharmonic confusions** — Pairs such as `A#maj`/`Bmaj` share identical pitch-class distributions and are easily confused by a chroma-based model. The confusion matrix will show elevated off-diagonal entries for these pairs.

3. **Context helps** — The context MLP consistently outperforms the single-frame baseline because chord identity depends partly on the temporal neighbourhood (e.g. chord transitions are smoother than random).

### Future Work

- **Recurrent / Transformer model** — capture long-range harmonic context beyond a fixed window.
- **Better features** — harmonic pitch-class profiles (HPCP) or CQT magnitude spectra are more discriminative than raw chroma.
- **Pitch-shift augmentation** — transposing audio by ±6 semitones multiplies the effective training set size by 12×.
- **HMM post-processing** — smooth the per-frame predictions with a transition model to reduce isolated incorrect frames.
